# Rimrock

__Au-Cu Deposit (Northeastern Nevada)__

## Summary

The Rimrock project is an ongoing geological study of a potential mineral deposit for a number of important metals (**Au, Cu**) and potentially other ore (Capps, 2011).

+ **Data**: Geochemistry, provided as elemental concentration (georeferenced soil samples).
+ **Goal**: Statistical analysis of the soil samples. Aim to determine statistical features of interest, especially as related to Au (gold ore) concentration.

## Data

The analysis uses the [`rb-rimrock`](https://github.com/rockybranches/rb-rimrock) package (private
repository; installed by `uv sync`). The data itself is **not** in this repository. Point the
notebook at it with environment variables before starting Jupyter:

| Variable | What |
|---|---|
| `RIMROCK_LOGS_DIR` | folder with the per-hole Excel drill logs (`R1.XLS`, `R3.xls`, …) |
| `RIMROCK_SOIL_SHP` | the surface soil shapefile (`rimrock_surface_data.shp` with its `.dbf`) |
| `RIMROCK_LOCATIONS_FILE_PATH` | collar table: the `Location` sheet of `RimrockDrillHoleData.xls`, or a CSV |

```bash
uv sync
RIMROCK_LOGS_DIR=~/rimrock/logs RIMROCK_SOIL_SHP=~/rimrock/soils/rimrock_surface_data.shp \
RIMROCK_LOCATIONS_FILE_PATH=~/rimrock/Rimrock_Drill_hole_data_Location.csv uv run jupyter lab
```

Depths are in **feet**. The formulas are in `rb-rimrock/docs/METHODS.md`.
Please commit this notebook **with outputs cleared**: this repository is public.

In [ ]:
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rb_rimrock.assays import hole_id_from_log, read_drill_log, read_locations
from rb_rimrock.config import get_settings
from rb_rimrock.correlation import au_weights
from rb_rimrock.holes import legacy_beta
from rb_rimrock.pipeline import build_hole, compute_report
from rb_rimrock import plots
from rb_rimrock.surface import cluster, surface_r2

settings = get_settings()
LOGS_DIR = Path(os.environ.get("RIMROCK_LOGS_DIR", settings.data_dir / "logs")).expanduser()
SOIL_SHP = Path(os.environ.get("RIMROCK_SOIL_SHP",
                               settings.surface_shapefile_dir / "rimrock_surface_data.shp")).expanduser()
COLLARS = settings.locations_file
CRS = settings.crs

# analysis options (see METHODS.md §3-§6)
GAPS = "interpolate"     # "nan" | "skip" | "interpolate"; R9 needs skip or interpolate
BETA = None              # None = legacy 4 / (pi * mean Au score)
INCLUDE_TARGET = True    # Au's own term in the per-sample weights
SEED = 0                 # surface k-means

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]  # validated, fixed order
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e4e3df", "grid.linewidth": 0.6, "axes.axisbelow": True})
print(LOGS_DIR, SOIL_SHP, COLLARS, sep="\n")

## Drill logs

Each log is parsed into one row per sampled interval (`rb_rimrock.assays.read_drill_log`):
the element columns after `DESCRIPTION`, `<limit` results as 0, `*`/`ND` as missing, and nothing
below the last Au assay. Workbooks without assays (the log-only `R8.XLS`) are skipped, and a hole
that appears twice keeps its first file.

In [ ]:
logs = {}
for path in sorted(p for p in LOGS_DIR.rglob("*") if p.suffix.lower() == ".xls"):
    try:
        df = read_drill_log(path)
    except ValueError:
        continue
    logs.setdefault(hole_id_from_log(path), df)

collars = read_locations(COLLARS).set_index("Bore")
summary = pd.DataFrame({
    "samples": {h: len(df) for h, df in logs.items()},
    "deepest sample (ft)": {h: df["DepthToTop"].max() for h, df in logs.items()},
    "TD (ft)": collars["TD"],
    "max Au (ppm)": {h: df["Au"].max() for h, df in logs.items()},
}).loc[sorted(logs)]
summary

## Per-sample Au weight by depth

For every sample, the co-enrichment of each element with Au is summed and rescaled to [0, 1]
(METHODS.md §3). `INCLUDE_TARGET` adds Au's own term.

In [ ]:
holes_ns = collars.loc[sorted(logs)].sort_values("Northing", ascending=False).index
fig, axes = plt.subplots(1, len(holes_ns), figsize=(2.3 * len(holes_ns), 7), sharey=True)
for ax, hole in zip(np.atleast_1d(axes), holes_ns):
    df = logs[hole]
    others = [c for c in df.columns if c not in ("Au", "DepthToTop")]
    w = au_weights(df, others, include_target=INCLUDE_TARGET)
    ax.barh(w["DepthToTop"], w["weights"], height=4.5, color=SERIES[0])
    ax.set_title(hole)
    ax.set_xlim(0, 1)
    ax.set_xlabel("weight")
np.atleast_1d(axes)[0].set_ylabel("depth to top (ft)")
np.atleast_1d(axes)[0].invert_yaxis()
fig.suptitle("Per-sample Au weight, holes north to south")
fig.tight_layout()

## Surface soils: R² with Au

Squared correlation of each normalized soil element with normalized Au (METHODS.md §5). Elements
that are constant over the samples (Th) have no R².

In [ ]:
soils = gpd.read_file(SOIL_SHP)
soils = soils if soils.crs is not None else soils.set_crs(CRS)
r2 = surface_r2(soils)
shown = r2.drop("Au").dropna().sort_values()
fig, ax = plt.subplots(figsize=(6, 7))
ax.barh(shown.index, shown.values, height=0.7, color=SERIES[0])
ax.set_xlabel("R² with Au (surface soils)")
ax.set_title(f"{len(soils):,} soil samples")
for y, v in enumerate(shown.values):
    if v >= 0.1:
        ax.text(v + 0.01, y, f"{v:.2f}", va="center", fontsize=8, color="#52514e")
fig.tight_layout()

## Per-hole Boltzmann weights

Depth-gradient scores per hole (METHODS.md §4), weighted across holes and by the surface R²
(§5), then a softmax per hole (§6). `GAPS` decides how depth gaps are handled and `BETA` sets the
softmax inverse temperature.

In [ ]:
holes = {h: build_hole(df, collars.loc[h], CRS) for h, df in logs.items()}
report = compute_report(holes, r2, gaps=GAPS, beta=BETA)
print(f"beta = {report.beta:,.1f}")
report.weights.loc[report.holes_order].round(3)

In [ ]:
plots.bar_figure(report.weights, report.holes_order)

## Sensitivity to β

The legacy β = 4 / (π · mean Au score) is large for these holes, which makes each hole's weights
nearly winner-take-all. The chart shows the largest weight in each hole as β falls (a line near
1 means one element takes all of the hole's weight); the table below it gives the values.

In [ ]:
betas = sorted({round(report.beta, 1), 3000, 1000, 300, 100, 30, 10}, reverse=True)
top = {}
for b in betas:
    w = compute_report(holes, r2, gaps=GAPS, beta=b).weights
    top[b] = pd.DataFrame({"top element": w.idxmax(axis=1), "weight": w.max(axis=1)})
max_w = pd.DataFrame({b: t["weight"] for b, t in top.items()}).loc[report.holes_order]

fig, ax = plt.subplots(figsize=(7, 4.5))
for color, hole in zip(SERIES, max_w.index):
    ax.plot(max_w.columns, max_w.loc[hole], color=color, linewidth=2, marker="o", markersize=5,
            label=hole)
ax.set_xscale("log")
ax.invert_xaxis()
ax.set_ylim(0, 1.05)
ax.set_xlabel("β (log scale, decreasing →)")
ax.set_ylabel("largest weight in the hole")
ax.axvline(report.beta, color="#8a8984", linewidth=1, linestyle="--")
ax.annotate("legacy β", (report.beta, 1.0), xytext=(4, 0), textcoords="offset points",
            fontsize=9, color="#52514e")
ax.legend(title="hole (N → S)", frameon=False, loc="upper right")
fig.tight_layout()

pd.concat({b: t["top element"] + " " + t["weight"].round(2).astype(str) for b, t in top.items()},
          axis=1).loc[report.holes_order]

## Gap policy

R9 has a split interval (572.5 and 575.5 ft), so its 5-ft neighbours are missing there. With
`"nan"` every R9 score is undefined; `"skip"` drops those points; `"interpolate"` fills the grid.

In [ ]:
by_policy = {}
for gaps in ("nan", "skip", "interpolate"):
    w = compute_report(holes, r2, gaps=gaps, beta=BETA).weights
    by_policy[gaps] = w.apply(lambda row: row.idxmax() if row.notna().any() else "— (all NaN)",
                              axis=1)
pd.DataFrame(by_policy).loc[report.holes_order]

## Map

Surface samples clustered with k-means (METHODS.md §7, shown as convex hulls) and the holes,
each with its pie of element weights when `rimrock report` has written the pie PNGs.

In [ ]:
located = soils.dropna(subset=["geometry"]).copy()
located["class"] = cluster(located, random_state=SEED)
plots.map_figure(report.locations, {}, located)

## References

+ [<img src="./imgs/Screenshot 2026-07-02 at 05-41-37 (PDF) Geologic Setting of Gold Mineralization at the Rimrock Prospect Western Sheep Creek Range Lander County Nevada.png" width="250"/>](Abstract.png)
    + [(Capps, et al., 2011), ResearchGate](https://www.researchgate.net/publication/289373035_Geologic_Setting_of_Gold_Mineralization_at_the_Rimrock_Prospect_Western_Sheep_Creek_Range_Lander_County_Nevada)